In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
import pandas as pd
from datetime import datetime as dt
from collections import Counter

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from utils.validator import (
    Validator,
    error_masks,
    warning_masks
)

today = dt.now()

roll_selection_start = dt(2025, 9, 30) # 1 day before october
roll_selection_end = dt(2025, 12, 10) # last day of october

# 31 October plans
# 38 November plans
# 45 December plans

In [3]:
COMPLEX_HRA_PATH = rf"G:\Shared drives\Development\Data\DEV Datasets\DEV-2938\Quicksight - Health_Plan_Codes_by_EmployerID - 20250919.csv"
complex_hra_df = pd.read_csv(COMPLEX_HRA_PATH)

In [4]:
complex_hra_df['org_external_identifier'].unique()

array(['RMRAAFCU', 'RMRABSA', 'RMRACF', 'RMRADCON', 'RMRAFA', 'RMRAHA',
       'RMRALR', 'RMRAMA', 'RMRAND', 'RMRARA', 'RMRARC', 'RMRARP',
       'RMRARS', 'RMRBAR', 'RMRBAYE', 'RMRBCS', 'RMRBCU', 'RMRBFR',
       'RMRBTP', 'RMRCCW', 'RMRCDA', 'RMRCEC', 'RMRCGRS', 'RMRCND',
       'RMRCOE', 'RMRCOPAC', 'RMRCPCD', 'RMRCSC', 'RMRCSD', 'RMRCSUF',
       'RMRDAR', 'RMRDCC', 'RMRDER', 'RMRDPAM', 'RMRDRM', 'RMRECH',
       'RMREFAA', 'RMRENV', 'RMREVT', 'RMRFBC', 'RMRFBP', 'RMRFED',
       'RMRFRM', 'RMRFSB', 'RMRGAS', 'RMRGCCI', 'RMRGCP', 'RMRGEM',
       'RMRGMW', 'RMRHAL', 'RMRHFK', 'RMRHOPE', 'RMRHSH', 'RMRIAC',
       'RMRIMD', 'RMRISI', 'RMRJFC', 'RMRJGL', 'RMRJMG', 'RMRJVA',
       'RMRKAV', 'RMRKCM', 'RMRLEIC', 'RMRLFR', 'RMRLOA', 'RMRMCW',
       'RMRMESA', 'RMRMHBC', 'RMRMHV', 'RMRMSC', 'RMRMVSD', 'RMRNGC',
       'RMRNOF', 'RMRNORA', 'RMRNPI', 'RMRNRB', 'RMRPBS', 'RMRPIP',
       'RMRPTG', 'RMRPWM', 'RMRRFTA', 'RMRRLE', 'RMRRLP', 'RMRRSD',
       'RMRSCR', 'RMRSHP', 'RMRSHR', 'RMR

In [5]:
''' open plans from previous notebook ''' # ./collect.ipynb

try:
    plan_df = pd.read_pickle(f"cache/{dt.strftime(today, "%y%m%d")}_ALL_PLANS_ORGS_CLIENTS.pkl")
    print(f"opened {len(plan_df)} plans")
except FileNotFoundError as e:
    print(f"{e}, please generate file in `collect.ipynb`")

opened 5848 plans


In [6]:
''' filter to plans that end between desired dates '''

filter_by_date_df = plan_df[
    (plan_df['plan_year.valid_to'] >= roll_selection_start) &
    (plan_df['plan_year.valid_to'] <= roll_selection_end)
]

print(f"{len(filter_by_date_df)} plans that end between {dt.strftime(roll_selection_start, "%m/%d/%Y")} and {dt.strftime(roll_selection_end, "%m/%d/%Y")}")

113 plans that end between 09/30/2025 and 12/10/2025


In [7]:
''' validate and add errors '''

valid_df = Validator(df=filter_by_date_df, error_masks=error_masks, warning_masks=warning_masks)
valid_df.validate()


''' AUXILLARY WARNINGS/ERRORS '''
# add warning to rows that have an organization that offers HRA's
hra_plans = valid_df[valid_df['account_type.account_type'] == 'HRA']
hra_org_ids = hra_plans['organization_id'].to_list()

hra_org_mask = valid_df['organization_id'].isin(hra_org_ids)
valid_df.add_warning(hra_org_mask, "org has HRA's")

# add errors to rows that have an organization that offer Complex HRA's
rmrcodes_w_complex_hras = complex_hra_df['org_external_identifier'].unique()
complex_hra_org_mask = valid_df['rmrcode'].isin(rmrcodes_w_complex_hras)
valid_df.add_error(complex_hra_org_mask, "skip orgs with complex HRA's")

''' check if plans already rolled '''
all_elevate_plans = pd.read_pickle(f"cache/{dt.strftime(today, "%y%m%d")}_ALL_PLANS_PROD.pkl")
all_clickup_plans = pd.read_pickle(f"cache/{dt.strftime(today, "%y%m%d")}_ALL_PLANS_CLICKUP.pkl")

elv_rolled_indices = []
cu_rolled_indices = []

for index, row in valid_df.iterrows():
    ''' check elevate plans '''
    elevate_match = all_elevate_plans[
        (all_elevate_plans['organization_id'] == row.get('organization_id')) &
        (all_elevate_plans['account_type.account_type'] == row.get('account_type.account_type')) &
        (all_elevate_plans['plan_year.valid_from'] > row.get('plan_year.valid_from'))
    ]

    if len(elevate_match) > 0:
        elv_rolled_indices.append(index)

    # ''' check clickup plans ''' - lol, nevermind
    # clickup_match = all_clickup_plans[
    #     (all_clickup_plans['name'].str.contains(row.get('rmrcode'))) &
    #     (all_clickup_plans['cu_account_type'] == row.get('cu_account_type')) &
    #     (all_clickup_plans['date_plan_start'] > row.get('plan_year.valid_from'))
    # ]

    # if len(clickup_match) > 0:
    #     print(f"\n---------------------------------------")
    #     print(f"{row.get('rmrcode')} {row.get('cu_account_type')} {row.get('date_plan_start')}")
    #     display(clickup_match[['name', 'date_plan_start', 'date_plan_end', 'cu_account_type']])
    #     print(f"---------------------------------------")

elv_rolled_mask = valid_df.index.isin(elv_rolled_indices)
valid_df.add_error(elv_rolled_mask, "future plan exists")

''' reporting '''

print(F"\n===================== ERRORS =====================")
all_errors = [error for error_list in valid_df['error'] for error in error_list]
errors = Counter(all_errors).most_common()
errors.sort(key=lambda x: x[1], reverse=True)

num_errors = 0
for err_tuple in errors:
    print(f"\t{err_tuple[0]:<30} {err_tuple[1]:>4}")
    num_errors += err_tuple[1]

num_err_rows = len(valid_df[valid_df['error'].apply(lambda x: len(x) > 0)])
print(f"\ttotal{num_errors:30}\n{"-"*50}\n\t{num_err_rows} / {len(filter_by_date_df)} errored plans ({num_err_rows/len(filter_by_date_df):>0.2%})\n")

print(F"\n==================== WARNINGS ====================")
all_warnings = [warning for warning_list in valid_df['warning'] for warning in warning_list]
warnings = Counter(all_warnings).most_common()
warnings.sort(key=lambda x: x[1], reverse=True)

num_warnings = 0
for warn_tuple in warnings:
    print(f"\t{warn_tuple[0]:<30} {warn_tuple[1]:>4}")
    num_warnings += warn_tuple[1]


num_warn_rows = len(valid_df[valid_df['warning'].apply(lambda x: len(x) > 0)])
print(f"\ttotal{num_warn_rows:30}\n{"-"*50}\n\t{num_err_rows} / {len(filter_by_date_df)} plans w warnings ({num_warn_rows/len(filter_by_date_df):>0.2%})\n")

valid_df.to_pickle(f"cache/{dt.strftime(today, '%y%m%d')}_FILTERED_PLANS.pkl")


===================== ERRORS =====================
	future plan exists               29
	skip orgs with complex HRA's      8
	non-active organization           7
	auto-renew not turned on          2
	short plan year                   1
	total                            47
--------------------------------------------------
	43 / 113 errored plans (38.05%)


==================== WARNINGS ====================
	org has HRA's                    18
	no `cu_plan_id` found             9
	hra                               8
	total                            23
--------------------------------------------------
	43 / 113 plans w warnings (20.35%)



In [ ]:
display(valid_df[valid_df['error'].apply(lambda x: 'no `cu_plan_id` found' in x )])

In [ ]:
display(valid_df[valid_df['error'].apply(lambda x: 'future plan exists' in x )]['plan_year.valid_from'].value_counts())
